# FX Analysis & Carry Trade Strategy
### Interest rate differentials, trend analysis, and carry trade backtesting

This notebook analyses **major FX pairs** using real data from **yfinance** and implements
a systematic **carry trade strategy** — one of the most studied strategies in FX markets.
The carry trade exploits interest rate differentials: borrow in low-yield currencies,
invest in high-yield currencies.

**FX analysis covered:**
- Trend analysis on major pairs (EUR/USD, USD/JPY, GBP/USD, AUD/USD)
- Rolling volatility, drawdown, and Sharpe ratio
- Cross-pair correlation matrix

**Carry trade strategy:**
- Central bank rate differentials as carry signal
- Long high-yield vs short low-yield currency pairs
- Performance: cumulative return, Sharpe, max drawdown, Calmar ratio
- Correlation with global risk appetite (S&P 500 proxy)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import yfinance as yf
import warnings

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11, "axes.titlesize": 13})
print("libraries imported successfully")

## 1. FX Data Download

We download daily spot rates for 6 major pairs via yfinance (expressed as units per USD or
standard quoting convention).

In [ ]:
FX_PAIRS = {
    "EURUSD=X": "EUR/USD",
    "USDJPY=X": "USD/JPY",
    "GBPUSD=X": "GBP/USD",
    "AUDUSD=X": "AUD/USD",
    "USDCHF=X": "USD/CHF",
    "NZDUSD=X": "NZD/USD",
}

START = "2018-01-01"

raw = yf.download(list(FX_PAIRS.keys()), start=START, auto_adjust=True, progress=False)["Close"]
raw = raw.rename(columns=FX_PAIRS).dropna(how="all").ffill()

print(f"Period    : {raw.index[0].date()} → {raw.index[-1].date()}")
print(f"Pairs     : {list(raw.columns)}")
print(f"Rows      : {len(raw)} trading days")
raw.tail(3)

## 2. Normalised Performance Overview

Base all pairs at 100 on the start date to compare cumulative appreciation.

In [ ]:
base = raw / raw.iloc[0] * 100

fig, ax = plt.subplots(figsize=(13, 5))
colors = plt.cm.tab10(np.linspace(0, 1, len(base.columns)))
for col, c in zip(base.columns, colors):
    ax.plot(base.index, base[col], linewidth=1.6, label=col, color=c)
ax.axhline(100, color="black", linewidth=0.8, linestyle="--")
ax.set_title("FX Pairs — Normalised Performance (Base 100)")
ax.set_ylabel("Index (Base = 100)")
ax.legend(ncol=3, fontsize=9)
plt.tight_layout()
plt.show()

## 3. Rolling Volatility (Annualised)

In [ ]:
log_ret = np.log(raw / raw.shift(1)).dropna()
roll_vol = log_ret.rolling(30).std() * np.sqrt(252) * 100

fig, ax = plt.subplots(figsize=(13, 5))
for col, c in zip(roll_vol.columns, colors):
    ax.plot(roll_vol.index, roll_vol[col], linewidth=1.3, label=col, color=c, alpha=0.85)
ax.set_title("30-Day Rolling Annualised Volatility (%)")
ax.set_ylabel("Volatility (%)")
ax.legend(ncol=3, fontsize=9)
plt.tight_layout()
plt.show()

## 4. Correlation Matrix

In [ ]:
corr = log_ret.corr()

fig, ax = plt.subplots(figsize=(8, 6))
cax = ax.matshow(corr, cmap="RdYlGn", vmin=-1, vmax=1)
fig.colorbar(cax)
pairs = list(corr.columns)
ax.set_xticks(range(len(pairs)))
ax.set_yticks(range(len(pairs)))
ax.set_xticklabels(pairs, rotation=45, ha="left", fontsize=10)
ax.set_yticklabels(pairs, fontsize=10)
for i in range(len(pairs)):
    for j in range(len(pairs)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center",
                fontsize=9, color="black" if abs(corr.iloc[i, j]) < 0.75 else "white")
ax.set_title("FX Log-Return Correlation Matrix", pad=20)
plt.tight_layout()
plt.show()

## 5. EUR/USD Deep Dive — Trend, Drawdown & Risk Metrics

In [ ]:
eurusd = raw["EUR/USD"].dropna()
ret_eu = np.log(eurusd / eurusd.shift(1)).dropna()

sma50  = eurusd.rolling(50).mean()
sma200 = eurusd.rolling(200).mean()

# Drawdown
cummax = eurusd.cummax()
drawdown = (eurusd - cummax) / cummax * 100

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True,
                          gridspec_kw={"height_ratios": [3, 1, 1]})

axes[0].plot(eurusd, color="black", linewidth=1.2, label="EUR/USD")
axes[0].plot(sma50,  color="steelblue", linewidth=1.8, linestyle="--", label="SMA 50")
axes[0].plot(sma200, color="coral",     linewidth=1.8, linestyle="--", label="SMA 200")
axes[0].set_title("EUR/USD — Price, Moving Averages & Drawdown")
axes[0].set_ylabel("EUR/USD")
axes[0].legend()

axes[1].fill_between(drawdown.index, drawdown, 0, color="red", alpha=0.5)
axes[1].set_ylabel("Drawdown (%)")

roll_ret = ret_eu.rolling(252).mean() * 252
roll_vol_eu = ret_eu.rolling(252).std() * np.sqrt(252)
roll_sharpe = roll_ret / roll_vol_eu
axes[2].plot(roll_sharpe, color="purple", linewidth=1.4)
axes[2].axhline(0, color="black", linewidth=0.8)
axes[2].set_ylabel("Rolling Sharpe")

plt.tight_layout()
plt.show()

ann_ret = ret_eu.mean() * 252
ann_vol = ret_eu.std() * np.sqrt(252)
max_dd  = drawdown.min()
print(f"EUR/USD ({START} → today)")
print(f"  Annualised return   : {ann_ret:.2%}")
print(f"  Annualised volatility: {ann_vol:.2%}")
print(f"  Sharpe ratio        : {ann_ret / ann_vol:.2f}")
print(f"  Max drawdown        : {max_dd:.1f}%")

## 6. Carry Trade Strategy

**Theory:** In an efficient market, the forward exchange rate should reflect interest rate
differentials (Covered Interest Parity). In practice, the **forward premium puzzle**
shows that high-yield currencies tend to *appreciate* rather than depreciate —
making the carry trade systematically profitable, with crash risk.

### Rate Differential Assumptions

We use approximate **central bank policy rates** by year to define the carry signal.
In practice, these come from Bloomberg/Refinitiv or central bank websites.

| Currency | Proxy rate (2023-2024) |
|----------|------------------------|
| USD      | ~5.25% (Fed)           |
| AUD      | ~4.35% (RBA)           |
| NZD      | ~5.50% (RBNZ)          |
| EUR      | ~4.00% (ECB)           |
| GBP      | ~5.25% (BoE)           |
| JPY      | ~0.10% (BoJ)           |
| CHF      | ~1.75% (SNB)           |

In [ ]:
# Approximate annualised short-term rates by currency
# Source: central bank policy rates (illustrative, static for simplicity)
RATES = {
    "USD": 0.0525,
    "EUR": 0.0400,
    "GBP": 0.0525,
    "AUD": 0.0435,
    "NZD": 0.0550,
    "JPY": 0.0010,
    "CHF": 0.0175,
}

# For each FX pair (expressed as quote/base), carry = rate(quote) - rate(base)
# Positive carry means the quote currency pays more — we want to be long USD when
# USD rates are higher than the foreign currency.
PAIR_CARRY = {
    "EUR/USD": RATES["EUR"] - RATES["USD"],   # negative → long USD vs EUR
    "USD/JPY": RATES["USD"] - RATES["JPY"],   # positive → long USD vs JPY
    "GBP/USD": RATES["GBP"] - RATES["USD"],   # ~0 → neutral
    "AUD/USD": RATES["AUD"] - RATES["USD"],   # negative → slight long USD
    "USD/CHF": RATES["USD"] - RATES["CHF"],   # positive → long USD vs CHF
    "NZD/USD": RATES["NZD"] - RATES["USD"],   # positive → long NZD
}

print("Carry differentials (annualised):")
for pair, carry in sorted(PAIR_CARRY.items(), key=lambda x: x[1], reverse=True):
    direction = "LONG" if carry > 0 else "SHORT"
    print(f"  {pair:12s}  carry = {carry:+.2%}  → {direction} first currency")

In [ ]:
# Build carry portfolio:
# Long pairs where carry > 0 (we earn the differential)
# Short pairs where carry < 0 (expressed by negating returns)
# Equal-weight across active legs

# Map pair names to log-return series
pair_returns = {}
for yf_ticker, name in FX_PAIRS.items():
    if name in log_ret.columns:
        pair_returns[name] = log_ret[name]

# Align all to a common date range
ret_df = pd.DataFrame(pair_returns).dropna()

# Carry sign: +1 if carry > 0 (long first ccy), -1 if carry < 0 (short first ccy)
carry_sign = {p: (1 if PAIR_CARRY.get(p, 0) > 0 else -1) for p in ret_df.columns}

# Daily carry return = sum of signed returns / number of active pairs
carry_daily = pd.Series(0.0, index=ret_df.index)
for pair, sign in carry_sign.items():
    if pair in ret_df.columns:
        # Add daily interest accrual (carry / 252) to the price return
        daily_carry_income = PAIR_CARRY.get(pair, 0) / 252 * sign
        carry_daily += (ret_df[pair] * sign + daily_carry_income)
carry_daily /= len(carry_sign)

cumul_carry = (1 + carry_daily).cumprod()

# Benchmark: equal-weight long-only FX basket
eq_long = ret_df.mean(axis=1)
cumul_eq = (1 + eq_long).cumprod()

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(cumul_carry.index, cumul_carry, color="steelblue", linewidth=2.2,
        label="Carry Trade Portfolio")
ax.plot(cumul_eq.index,    cumul_eq,    color="coral",     linewidth=2.0,
        linestyle="--", label="Equal-Weight FX Basket")
ax.axhline(1, color="black", linewidth=0.8, linestyle="--")
ax.set_title("FX Carry Trade vs Equal-Weight Basket — Cumulative Return")
ax.set_ylabel("Growth of $1")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Performance Attribution

In [ ]:
def perf_metrics(ret: pd.Series, label: str, rf: float = 0.0) -> dict:
    ann_r  = ret.mean() * 252
    ann_v  = ret.std() * np.sqrt(252)
    sr     = (ann_r - rf) / ann_v
    cumul  = (1 + ret).cumprod()
    mdd    = float(((cumul / cumul.cummax()) - 1).min())
    calmar = ann_r / abs(mdd) if mdd != 0 else np.nan
    return {"Strategy": label, "Ann. Return": f"{ann_r:.2%}",
            "Ann. Vol": f"{ann_v:.2%}", "Sharpe": f"{sr:.2f}",
            "Max DD": f"{mdd:.2%}", "Calmar": f"{calmar:.2f}"}

rows = [perf_metrics(carry_daily, "Carry Trade"),
        perf_metrics(eq_long,     "Equal-Weight FX")]

# Add individual pairs
for pair in ret_df.columns:
    rows.append(perf_metrics(ret_df[pair], pair))

df_perf = pd.DataFrame(rows).set_index("Strategy")
print(df_perf.to_string())

## 8. Carry Trade & Risk Appetite — Correlation with Equities

The carry trade is known to be a **risk-on** strategy: it performs well in calm
markets and suffers sharp reversals during risk-off episodes (2008, March 2020).

In [ ]:
# Download S&P 500 as risk proxy
spy = yf.download("^GSPC", start=START, auto_adjust=True, progress=False)["Close"]
spy_ret = np.log(spy / spy.shift(1)).dropna().rename("S&P 500")

# Align
df_risk = pd.DataFrame({
    "Carry Trade": carry_daily,
    "S&P 500":     spy_ret,
}).dropna()

rolling_corr = df_risk["Carry Trade"].rolling(90).corr(df_risk["S&P 500"])

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True,
                          gridspec_kw={"height_ratios": [2, 2, 1]})

cumul_carry_aligned = (1 + df_risk["Carry Trade"]).cumprod()
cumul_spy           = (1 + df_risk["S&P 500"]).cumprod()

axes[0].plot(cumul_carry_aligned.index, cumul_carry_aligned,
             color="steelblue", linewidth=2, label="Carry Trade")
axes[0].set_title("Carry Trade vs S&P 500 — Cumulative Performance & Rolling Correlation")
axes[0].set_ylabel("Growth of $1")
axes[0].legend()

axes[1].plot(cumul_spy.index, cumul_spy, color="coral", linewidth=2, label="S&P 500")
axes[1].set_ylabel("Growth of $1")
axes[1].legend()

axes[2].plot(rolling_corr.index, rolling_corr,
             color="purple", linewidth=1.5)
axes[2].axhline(0, color="black", linewidth=0.8)
axes[2].fill_between(rolling_corr.index, rolling_corr, 0,
                     where=(rolling_corr >= 0), alpha=0.3, color="green")
axes[2].fill_between(rolling_corr.index, rolling_corr, 0,
                     where=(rolling_corr < 0),  alpha=0.3, color="red")
axes[2].set_ylabel("90D Rolling Corr")
axes[2].set_ylim(-1, 1)

plt.tight_layout()
plt.show()

corr_full = df_risk.corr().loc["Carry Trade", "S&P 500"]
print(f"Full-period correlation: Carry Trade vs S&P 500 = {corr_full:.3f}")

## 9. Carry Trade Drawdown Analysis

In [ ]:
cumul = (1 + carry_daily).cumprod()
dd    = (cumul / cumul.cummax() - 1) * 100

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True,
                          gridspec_kw={"height_ratios": [2, 1]})
axes[0].plot(cumul, color="steelblue", linewidth=2)
axes[0].set_title("Carry Trade — Cumulative Return & Drawdown")
axes[0].set_ylabel("Growth of $1")

axes[1].fill_between(dd.index, dd, 0, color="red", alpha=0.5)
axes[1].set_ylabel("Drawdown (%)")
axes[1].set_xlabel("")

plt.tight_layout()
plt.show()

print(f"Maximum drawdown : {dd.min():.1f}%")
print(f"Current drawdown : {dd.iloc[-1]:.1f}%")